<a href="https://colab.research.google.com/github/vieer-dwivedi/SwamivivekanadBook-Rag-GenAI/blob/main/swamivivekanad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements




In [ ]:
!pip install pymupdf  sentence-transformers lancedb pyarrow transformers chromadb

## Stream Book

In [ ]:
import fitz
import requests
remote = requests.get("https://www.vedanta-pitt.org/wp-content/uploads/2020/05/Complete_Works_of_Swami_Vivekananda_all_volumes.pdf")
doc = fitz.Document(
    stream=remote.content,
    filetype="pdf"
    )


# Document Structured Chunking

In [ ]:
toc = doc.get_toc()

sections = []

for i, item in enumerate(toc):

    level, title, start_page = item

    if i < len(toc) - 1:
        end_page = toc[i + 1][2] - 1
    else:
        end_page = len(doc)

    section = {
        "level": level,
        "title": title,
        "start_page": start_page,
        "end_page": end_page,
        "subheadings": []
    }

    if level >= 2:

        current_subheading = None

        for page_num in range(
            start_page - 1,
            end_page
        ):

            page = doc[page_num]

            blocks = page.get_text("dict")["blocks"]

            for block in blocks:

                if "lines" not in block:
                    continue

                text = " ".join(
                    span["text"]
                    for line in block["lines"]
                    for span in line["spans"]
                ).strip()

                if not text:
                    continue

                first_span = block["lines"][0]["spans"][0]

                is_heading = (
                    "Bold" in first_span["font"]
                    and len(text) < 120
                    and text != title
                )

                if is_heading:

                    current_subheading = {
                        "title": text,
                        "page": page_num + 1,
                        "font": first_span["font"],
                        "size": first_span["size"],
                        "paragraphs": []
                    }

                    section["subheadings"].append(
                        current_subheading
                    )

                else:

                    if current_subheading:

                        lines_count = len(block["lines"])

                        x0, y0, x1, y1 = block["bbox"]
                        width = x1 - x0
                        is_real_paragraph = not (
                            lines_count < 3
                            and width < 300
                        )

                        if is_real_paragraph:

                            current_subheading[
                                "paragraphs"
                            ].append({
                                "page": page_num + 1,
                                "text": text,
                                "lines": lines_count,
                                "width": width
                            })

    sections.append(section)

## Print Structure

In [ ]:
for section in sections[:5]:

    print("\n" + "=" * 120)

    print(f"📁 {section['title']}")

    for sub in section["subheadings"]:

        print(f"   └── 📂 {sub['title']}")

        for i, para in enumerate(
            sub["paragraphs"][:3],
            start=1
        ):

            preview = para["text"][:120].replace(
                "\n",
                " "
            )

            print(
                f"       └── 📄 P{i}: "
                f"{preview}..."
            )

        if len(sub["paragraphs"]) > 3:

            print(
                f"       └── ... "
                f"({len(sub['paragraphs'])} paragraphs total)"
            )

## Add Meta Data to Paragraphs

In [ ]:
paragraphs = []

for doc in documents:

    for para in doc["paragraphs"]:

        paragraphs.append({

            "text": para["text"],

            "metadata": {
                "source": doc["source"],
                "volume": doc["volume"],
                "section": doc["section"],
                "subheading": doc["subheading"],
                "page": para["page"],
                "citation": para["citation"]
            }
        })

print(
    f"Total paragraphs: {len(paragraphs)}"
)

In [ ]:
documents = []

current_volume = None

for section in sections:

    if section["level"] == 1:
        current_volume = section["title"]
        continue

    for sub in section["subheadings"]:

        enriched_paragraphs = []

        for idx, para in enumerate(
            sub["paragraphs"],
            start=1
        ):

            enriched_paragraphs.append({

                "paragraph_id": idx,

                "page": para["page"],

                "lines": para["lines"],

                "width": para["width"],

                "char_count": len(
                    para["text"]
                ),

                "word_count": len(
                    para["text"].split()
                ),

                "citation":
                f"{current_volume} > "
                f"{section['title']} > "
                f"{sub['title']} > "
                f"Page {para['page']}",

                "text": para["text"]
            })

        documents.append({

            "source":
            "Complete Works of Swami Vivekananda",

            "volume":
            current_volume,

            "section":
            section["title"],

            "subheading":
            sub["title"],

            "paragraph_count":
            len(enriched_paragraphs),

            "paragraphs":
            enriched_paragraphs
        })

# Semantic Chunks

## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

texts = [
    p["text"]
    for p in paragraphs
]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

print(
    f"Embeddings: {len(embeddings)}"
)

## Similarties

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = []

for i in range(len(embeddings) - 1):

    score = cosine_similarity(
        [embeddings[i]],
        [embeddings[i + 1]]
    )[0][0]

    similarities.append(score)

print(
    f"Similarities: {len(similarities)}"
)

## Tokenizing

In [ ]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-small-en-v1.5"
)


THRESHOLD = 0.70
MAX_TOKENS = 500
OVERLAP_PARAGRAPHS = 1

semantic_chunks = []

current_chunk = [paragraphs[0]]

current_tokens = len(
    tokenizer.encode(
        paragraphs[0]["text"],
        add_special_tokens=False
    )
)

for i, score in enumerate(similarities):

    next_para = paragraphs[i + 1]

    next_tokens = len(
        tokenizer.encode(
            next_para["text"],
            add_special_tokens=False
        )
    )

    # ======================================
    # CASE 1: SINGLE PARAGRAPH > LIMIT
    # ======================================

    if next_tokens > MAX_TOKENS:

        semantic_chunks.append(
            current_chunk
        )

        semantic_chunks.append(
            [next_para]
        )

        current_chunk = []
        current_tokens = 0

        continue

    # ======================================
    # CASE 2: SAME TOPIC + WITHIN LIMIT
    # ======================================

    if (
        score >= THRESHOLD
        and
        current_tokens + next_tokens <= MAX_TOKENS
    ):

        current_chunk.append(
            next_para
        )

        current_tokens += next_tokens

    else:

        semantic_chunks.append(
            current_chunk
        )

        # ==================================
        # SAFE OVERLAP
        # ==================================

        overlap = []

        overlap_tokens = 0

        for para in reversed(
            current_chunk
        ):

            para_tokens = len(
                tokenizer.encode(
                    para["text"],
                    add_special_tokens=False
                )
            )

            if (
                overlap_tokens
                + para_tokens
                + next_tokens
                <= MAX_TOKENS
            ):

                overlap.insert(
                    0,
                    para
                )

                overlap_tokens += (
                    para_tokens
                )

            else:

                break

        current_chunk = (
            overlap
            + [next_para]
        )

        current_tokens = (
            overlap_tokens
            + next_tokens
        )

if current_chunk:

    semantic_chunks.append(
        current_chunk
    )

print(
    f"Semantic Chunks: "
    f"{len(semantic_chunks)}"
)

## Print Tokens Details

In [ ]:
token_counts = []

for chunk in semantic_chunks:

    text = "\n\n".join(
        p["text"]
        for p in chunk
    )

    count = len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )

    token_counts.append(count)

print("Min:", min(token_counts))
print("Max:", max(token_counts))
print(
    "Chunks >500:",
    sum(
        1
        for x in token_counts
        if x > 500
    )
)

## Print Paragraph Details

In [ ]:
large_paragraphs = []

for p in paragraphs:

    count = len(
        tokenizer.encode(
            p["text"],
            add_special_tokens=False
        )
    )

    if count > 500:

        large_paragraphs.append(count)

print(
    "Paragraphs >500:",
    len(large_paragraphs)
)

print(
    "Largest paragraph:",
    max(large_paragraphs)
)

# Vector Database

In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="./vivekananda_db"
)

## Create Collection

In [ ]:
collection = client.get_or_create_collection(
    name="vivekananda"
)

## Finalize Chunks

In [ ]:
final_chunks = []

for idx, chunk in enumerate(
    semantic_chunks,
    start=1
):

    if len(chunk) == 0:
        continue

    text = "\n\n".join(
        p["text"]
        for p in chunk
    )

    final_chunks.append({

        "id": f"chunk_{idx}",

        "text": text,

        "volume":
        chunk[0]["metadata"]["volume"],

        "section":
        chunk[0]["metadata"]["section"],

        "subheading":
        chunk[0]["metadata"]["subheading"],

        "pages": ",".join(
            map(
                str,
                sorted(
                    set(
                        p["metadata"]["page"]
                        for p in chunk
                    )
                )
            )
        )
    })

print(
    f"Final chunks: "
    f"{len(final_chunks)}"
)

## Embeding and Pushing to Vector DB

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)
BATCH_SIZE = 100

for i in range(
    0,
    len(final_chunks),
    BATCH_SIZE
):

    batch = final_chunks[
        i:i+BATCH_SIZE
    ]

    texts = [
        x["text"]
        for x in batch
    ]

    embeddings = model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    collection.add(

        ids=[
            x["id"]
            for x in batch
        ],

        documents=texts,

        embeddings=
        embeddings.tolist(),

        metadatas=[
            {
                "volume":
                x["volume"],

                "section":
                x["section"],

                "subheading":
                x["subheading"],

                "pages":
                x["pages"]
            }

            for x in batch
        ]
    )

print(
    f"Stored "
    f"{collection.count()} chunks"
)

print(
    collection.count()
)

# LLM Integration with Rag

## Retrieve Context

In [ ]:
def retrieve_context(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

## Build Context

In [ ]:
def build_context(
    results
):

    context_parts = []

    for doc, meta in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):

        context_parts.append(
            f"""
Volume: {meta.get('volume','')}

Section: {meta.get('section','')}

Pages: {meta.get('pages','')}

Content:
{doc}
"""
        )

    return "\n\n".join(
        context_parts
    )

## Create Prompt

In [ ]:
def ask_llm(
    query,
    context
):

    messages = [
        {
            "role": "system",
            "content": """
You are a helpful RAG assistant.

Answer ONLY from the provided context.

If the answer is not available
in the context, say:

'I could not find the answer in the provided documents.'
"""
        },
        {
            "role": "user",
            "content": f"""
Context:

{context}

Question:

{query}
"""
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8000
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

## LLM

### Load Model

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
import torch

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

### Rag Call Function

In [ ]:
def rag(
    query,
    top_k=5
):

    results = retrieve_context(
        query,
        top_k
    )

    context = build_context(
        results
    )

    answer = ask_llm(
        query,
        context
    )

    return answer

### Test

In [ ]:
query = (
    "When did Swami Vivekananda first arrive in America?"
)

answer = rag(
    query,
    top_k=10
)

print(answer)